#**Introduction - purpose of EDA and what the notebook covers**

## 🔍 Exploratory Data Analysis (EDA)
## Food Price Forecasting for Stable Commodities in Nigeria
### TS Academy Data Science Capstone 2026

---

## Purpose of This Notebook

This notebook conducts a comprehensive Exploratory Data Analysis on the
master dataset produced by the Data Cleaning and Merging pipeline.

EDA serves as the critical bridge between data preparation and modelling.
Before any model is built, we must deeply understand the data — its
structure, patterns, relationships, anomalies and limitations. The insights
discovered here directly inform every modelling decision that follows.

## What This Notebook Covers

This notebook is organised into eight sections:

1. **Data Loading and Verification** — confirm the dataset loaded correctly
   and matches expected dimensions
2. **Univariate Analysis** — understand the distribution of each variable
   individually
3. **Time Series Analysis** — examine price trends, seasonality,
   stationarity and autocorrelation over time
4. **Bivariate Analysis** — explore relationships between features
   and the target variable price_ngn
5. **Multivariate Analysis** — compare prices across states, commodities
   and key economic events
6. **Outlier Detection** — identify extreme values and data coverage gaps
7. **Key Findings** — summarise the most important discoveries from
   the analysis
8. **Recommendations for Modelling Team** — translate EDA findings into
   actionable guidance for model building

## Dataset Being Analysed

The master dataset (master_dataset_clean.csv) contains 2,846 rows and
18 columns covering 13 Nigerian states, 7 commodity variants and monthly
observations from January 2017 to December 2024. It integrates 8 data
sources including food prices, inflation, exchange rates, monetary policy,
rainfall, fuel prices, import volumes and conflict events.

#**STEP 1 - Data Loading and Verification**

## Purpose
Before any analysis begins we must confirm that the master dataset has
loaded correctly and matches the expected dimensions documented during
the merging phase. This step catches any issues early — before they
silently affect downstream analysis.

## What We Are Checking
- Dataset dimensions match expected 2,846 rows and 18 columns
- Date column is correctly parsed as datetime dtype
- All 13 states are present with no unexpected additions or omissions
- All 7 commodity variants are present and correctly named
- Null values only exist in the expected lag columns — price_lag1,
  price_lag2 and price_lag3
- Zero duplicate rows exist in the dataset
- Date range spans January 2017 to December 2024
- Numeric columns have sensible ranges — no negative prices,
  no impossibly large values

## Libraries Used in This Notebook
| Library | Purpose |
|---------|---------|
| pandas | Data loading, manipulation and aggregation |
| numpy | Numerical computations and array operations |
| matplotlib.pyplot | Creating all charts and visualisations |
| matplotlib.dates | Formatting date labels on chart axes |
| seaborn | Statistical visualisations and heatmaps |
| scipy.stats | Statistical testing and hypothesis tests |
| warnings | Suppressing non-critical library warnings |

## Visual Style
All charts in this notebook use the seaborn-v0_8-whitegrid style —
a clean white background with subtle grey gridlines that produces
professional publication-ready visualisations consistently across
all plots without needing to style each chart individually.

In [2]:
#Mounting my Drive
#connect my drive to colab
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
#importing of libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')

In [4]:
#reading in our master dataset
df = pd.read_csv('/content/drive/MyDrive/price forecasting project data(cleaned)/master_dataset_clean.csv')

#previewing the dataset
df.head()

,date,state,commodity,price_ngn,rainfall_mm,pms_price_ngn,conflict_events,conflict_score_weighted,conflict_fatalities,inflation_rate_pct,exchange_rate_ngn_usd,mpr_pct,import_volume_tonnes,price_lag1,price_lag2,price_lag3,harvest_season,other_commodity_avg_price
0,2021-03-01,Abia,Maize (white),280.52,0.0,158.57,8.0,33.0,6.0,46.082248,381.000000,11.5,98313.0,NaN,NaN,NaN,0,617.853333
1,2021-05-01,Abia,Maize (white),345.27,0.0,166.13,13.0,47.0,8.0,47.029667,400.126667,11.5,98313.0,280.52,NaN,NaN,0,779.354000
2,2021-07-01,Abia,Maize (white),400.30,0.0,182.23,1.0,1.0,0.0,47.961686,410.121000,11.5,98313.0,345.27,280.52,NaN,0,825.926667
3,2021-08-01,Abia,Maize (white),294.04,0.0,171.51,7.0,25.0,2.0,48.470936,410.385455,11.5,98313.0,400.30,345.27,280.52,1,810.984000
4,2021-10-01,Abia,Maize (white),333.88,0.0,170.25,9.0,33.0,6.0,49.526367,411.247368,11.5,98313.0,294.04,400.30,345.27,1,798.300000


In [11]:
#basic checks to confirm the dataset loaded correctly
print(df.shape) # to confirm 2846 rows and 18 columns
print('')
print(df.dtypes) # to confirm date is datetime
print('')
print(df.isnull().sum()) # confirm only lag columns have nulls
print('')
print(df.duplicated().sum()) # confirm zero duplicates
print('')
print(df['state'].unique()) # confirm 13 states
print('')
print(df['commodity'].unique()) # confirm 7 commodities




(2846, 18)

date                          object
state                         object
commodity                     object
price_ngn                    float64
rainfall_mm                  float64
pms_price_ngn                float64
conflict_events              float64
conflict_score_weighted      float64
conflict_fatalities          float64
inflation_rate_pct           float64
exchange_rate_ngn_usd        float64
mpr_pct                      float64
import_volume_tonnes         float64
price_lag1                   float64
price_lag2                   float64
price_lag3                   float64
harvest_season                 int64
other_commodity_avg_price    float64
dtype: object

date                           0
state                          0
commodity                      0
price_ngn                      0
rainfall_mm                    0
pms_price_ngn                  0
conflict_events                0
conflict_score_weighted        0
conflict_fatalities            0
inflation_

In [15]:
#converting date from 'object' to 'period['M']'
df['date'] = pd.to_datetime(df['date']).dt.to_period('M')

#check
print(df.dtypes)

date                         period[M]
state                           object
commodity                       object
price_ngn                      float64
rainfall_mm                    float64
pms_price_ngn                  float64
conflict_events                float64
conflict_score_weighted        float64
conflict_fatalities            float64
inflation_rate_pct             float64
exchange_rate_ngn_usd          float64
mpr_pct                        float64
import_volume_tonnes           float64
price_lag1                     float64
price_lag2                     float64
price_lag3                     float64
harvest_season                   int64
other_commodity_avg_price      float64
dtype: object


In [16]:
#check
df.head()

,date,state,commodity,price_ngn,rainfall_mm,pms_price_ngn,conflict_events,conflict_score_weighted,conflict_fatalities,inflation_rate_pct,exchange_rate_ngn_usd,mpr_pct,import_volume_tonnes,price_lag1,price_lag2,price_lag3,harvest_season,other_commodity_avg_price
0,2021-03,Abia,Maize (white),280.52,0.0,158.57,8.0,33.0,6.0,46.082248,381.000000,11.5,98313.0,NaN,NaN,NaN,0,617.853333
1,2021-05,Abia,Maize (white),345.27,0.0,166.13,13.0,47.0,8.0,47.029667,400.126667,11.5,98313.0,280.52,NaN,NaN,0,779.354000
2,2021-07,Abia,Maize (white),400.30,0.0,182.23,1.0,1.0,0.0,47.961686,410.121000,11.5,98313.0,345.27,280.52,NaN,0,825.926667
3,2021-08,Abia,Maize (white),294.04,0.0,171.51,7.0,25.0,2.0,48.470936,410.385455,11.5,98313.0,400.30,345.27,280.52,1,810.984000
4,2021-10,Abia,Maize (white),333.88,0.0,170.25,9.0,33.0,6.0,49.526367,411.247368,11.5,98313.0,294.04,400.30,345.27,1,798.300000


In [10]:
#get the summary statistics
print(df.describe())

         price_ngn  rainfall_mm  pms_price_ngn  conflict_events  \
count  2846.000000  2846.000000    2846.000000      2846.000000   
mean    848.757275    79.204202     205.018257        14.770555   
std     885.990718   115.320231     190.648470        18.352142   
min      30.000000     0.000000      44.560000         0.000000   
25%     210.055000     0.000000     144.750000         2.000000   
50%     631.774091     8.600000     158.570000         6.000000   
75%    1196.943750   132.100000     177.360000        21.000000   
max    6358.422857   551.200000    1283.790000        86.000000   

       conflict_score_weighted  conflict_fatalities  inflation_rate_pct  \
count              2846.000000          2846.000000         2846.000000   
mean                 59.912157            58.944483           47.284574   
std                  80.108911            97.822649           17.321427   
min                   0.000000             0.000000           23.746418   
25%                  

## Summary Statistics Interpretation

The summary statistics below were generated using df.describe() and
provide a statistical overview of all numeric columns in the master
dataset. Beyond simple inspection, summary statistics serve five
purposes in this analysis:

1. **Outlier detection** — identifying extreme values that may
   affect model performance
2. **Distribution shape** — large gaps between mean and median
   indicate skewed distributions
3. **Scale awareness** — features operate at very different scales
   (Naira vs percentages vs millimetres) which informs feature
   scaling decisions for the modelling team
4. **Data sanity check** — confirming no impossible values exist
   such as negative prices or zero rainfall everywhere
5. **Feature variance** — high standard deviation relative to mean
   indicates high variability which can be a strong predictive signal

---

### Key Observations by Column

**price_ngn — Target Variable**
Mean of ₦848 with a standard deviation of ₦885 — the STD being
almost as large as the mean signals extremely high price variation
across commodities and states. The interquartile range spans ₦210
at the 25th percentile to ₦1,196 at the 75th percentile — a
nearly ₦1,000 gap — confirming that cheap commodities like Maize
and expensive ones like Yam and Rice occupy very different price
ranges within the same dataset. Minimum of ₦30 and maximum of
₦6,358 are genuine values reflecting the full spectrum of
commodity prices across 8 years.

**rainfall_mm**
Mean of 79mm with a standard deviation of 115mm and a minimum of
zero. The 25th percentile is also zero — meaning at least 25% of
all state-month observations recorded no rainfall at all. This
reflects the dry season reality of northern states like Kano,
Katsina, Kebbi and Zamfara where months of zero rainfall are
normal. The maximum of 551mm reflects peak wet season rainfall
in southern states. The distribution is heavily right-skewed and
this will be clearly visible in the histogram.

**pms_price_ngn — Petrol Price**
Mean of ₦205 with a median of only ₦158 and a 75th percentile of
₦177 — meaning three quarters of all observations fall below ₦178.
This confirms that the vast majority of the dataset covers the
pre-subsidy removal era when petrol prices were heavily controlled
and artificially low. Only the top 25% of observations captures
the dramatic post-June 2023 price surge which pushed prices to a
maximum of ₦1,283. This column will show one of the most dramatic
structural breaks in the entire dataset during time series analysis.

**conflict_events**
Mean of 14 events per state-month with a minimum of zero and median
of 6. The zero minimum and low 25th percentile of 2 confirm that
conflict is concentrated in specific states and periods rather than
uniformly distributed across the dataset. Maximum of 86 events in
a single state-month reflects the most intense conflict periods in
Borno, Zamfara and Yobe.

**conflict_score_weighted**
Mean weighted score of 59 with standard deviation of 80 — the STD
exceeding the mean confirms a right-skewed distribution. The maximum
of 403 represents months of intense high-severity conflict. The
wide gap between median (23) and mean (59) further confirms that
extreme conflict events pull the mean significantly upward.

**conflict_fatalities**
Mean of 58 fatalities per state-month with a median of 8 — meaning
in a typical conflict month at least 8 people died. The maximum of
580 fatalities in a single state-month reflects the most severe
insurgency events in the Northeast. The heavily right-skewed
distribution with standard deviation of 97 indicates that extreme
fatality events are rare but significant enough to pull the mean
far above the median.

**inflation_rate_pct**
Note — the _pct suffix confirms values are expressed as percentages.
The minimum value of 23.75% is particularly significant — it means
Nigerian food inflation never dropped below 23% during the entire
2017 to 2024 study period. There was no low inflation era in this
dataset. The mean of 47% and maximum of 115% reflect the severe
inflationary environment that intensified from 2022 onwards. The
relatively moderate standard deviation of 17% suggests persistent
elevated inflation rather than isolated spikes.

**exchange_rate_ngn_usd**
The most structurally disrupted feature in the dataset. The 25th
percentile of ₦306 and 75th percentile of ₦415 show relative
stability for most of the study period — but the maximum of ₦1,670
reveals a catastrophic devaluation concentrated in the final period
of the dataset. The gap between the 75th percentile (₦415) and the
maximum (₦1,670) is far larger than any other feature proportionally,
confirming a sudden structural break rather than gradual depreciation.
Significant outliers are expected and entirely genuine.

**mpr_pct**
Note — the _pct suffix confirms values are expressed as percentages.
Mean of 13.78% with a standard deviation of only 3.4% and a range
of 11.5% to 27.5% — the least volatile feature in the entire
dataset. The CBN adjusts the Monetary Policy Rate infrequently
and in measured increments which explains the tight clustering.
The maximum of 27.5% reflects the aggressive rate hikes of 2024
as the CBN responded to runaway inflation.

**import_volume_tonnes**
Mean of 36,152 tonnes with a standard deviation of 54,887 tonnes —
the STD is 1.5 times the mean, confirming extreme dispersion. The
25th percentile of only 10.79 tonnes versus the 75th percentile of
98,313 tonnes represents the most dramatic interquartile jump of
any feature. This reflects the fundamental difference between
heavily imported commodities like Rice which records hundreds of
thousands of tonnes and domestically produced commodities like Yam
and Tomatoes which record near-zero import volumes. These are
genuine values not outliers.

**price_lag1, price_lag2, price_lag3**
All three lag features closely mirror the distribution of price_ngn
with means of ₦835, ₦820 and ₦805 respectively — a slight downward
trend across lags reflecting that prices have generally risen over
the study period, so older prices are on average slightly lower than
current prices. The close alignment between lag distributions and
price_ngn validates that the lag calculation was performed correctly.
Note that price_lag1 has 2,775 non-null values, price_lag2 has
2,705 and price_lag3 has 2,635 — the reduction reflects the
expected warmup period at the start of each state-commodity series.

**harvest_season**
Binary column with mean of 0.255 — confirming that approximately
25.5% of all rows represent harvest months. This aligns precisely
with the expected distribution of 2 to 3 harvest months per
commodity per 12-month cycle. The equal minimum of 0 and maximum
of 1 confirm the binary encoding is correct.

**other_commodity_avg_price**
Mean of ₦848 — identical to price_ngn mean, which is mathematically
expected since this feature is derived by averaging price_ngn across
commodities within the same state and month. Standard deviation of
₦618 with a maximum of ₦5,083 reflects cross-commodity price
pressure during the most intense inflationary periods of 2023
and 2024.

---

### Overall Assessment

The dataset displays the expected statistical characteristics of
Nigerian food price data across a period of significant economic
disruption. Key patterns identified:

- **High variance features** — price_ngn, import_volume_tonnes,
  exchange_rate_ngn_usd and conflict_fatalities all show standard
  deviations at or exceeding their means
- **Structural break features** — pms_price_ngn and
  exchange_rate_ngn_usd show dramatic jumps concentrated in
  2023 that will require careful handling during modelling
- **Zero-inflated features** — rainfall_mm, conflict_events and
  conflict_fatalities all have 25th percentiles at or near zero
- **Stable features** — mpr_pct shows the tightest distribution
  of all features reflecting deliberate policy rate management
- **No data integrity issues** — no negative prices, no impossible
  values and null values only in the expected lag warmup rows

The dataset is confirmed statistically sound and ready for
deeper exploratory analysis.

#**STEP 2 - Univariate Analysis**

#**STEP 3 - Time Series Analysis**

#**STEP 4 - Bivariate Analysis**

#**STEP 5 - Multivariate Analysis**

#**STEP 6 - Outlier Detection**

#**STEP 7 - Key Findings Summary**

#**STEP 8 - Recommendations for Modelling Team**